In [3]:
import bpy
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

if (0):
    utils.remove_libraries()
    utils.clean_scene()
else:
    filepath = './scenes/cornell_box.blend'
    bpy.ops.wm.open_mainfile(filepath=filepath)

    obj = bpy.data.objects["light"]
    bpy.data.objects.remove(obj)

    cam = bpy.context.scene.camera
    cam.data.clip_start = 1e-6
    cam.keyframe_insert(data_path="location", frame=1)

    # Add light source
    bpy.ops.object.light_add(type='POINT')
    light = bpy.data.objects["Point"]
    light.location = cam.location
    light.data.energy = 1e9

00:47.194  blend            | Read blend: "/Users/motoole/git/computational-imaging-lessons/scenes/cornell_box.blend"


In [4]:
if (0):
    glossy_material = bpy.data.materials.new(name="GlossyMaterial")
    glossy_material.use_nodes = True

    bpy.ops.mesh.primitive_plane_add(size=4, location=(0, 5, 0), rotation=(np.pi/2, 0, np.pi/4))
    plane = bpy.context.active_object
    plane.data.materials.append(glossy_material)

    bpy.ops.mesh.primitive_plane_add(size=4, location=(0, 5, 0), rotation=(0, np.pi/2, np.pi/4))
    plane = bpy.context.active_object
    plane.data.materials.append(glossy_material)

In [5]:
# Set render engine for ray tracing
bpy.context.scene.render.engine = 'CYCLES'
bpy.context.scene.render.resolution_x = 256
bpy.context.scene.render.resolution_y = 256
bpy.context.scene.render.image_settings.file_format = 'OPEN_EXR'
bpy.context.scene.view_settings.view_transform = 'Raw'
bpy.context.scene.cycles.use_adaptive_sampling = False
bpy.context.scene.cycles.samples = 1
bpy.context.scene.cycles.use_denoising = False
bpy.context.scene.cycles.sample_clamp_indirect = 0

In [6]:
node_group = bpy.data.node_groups.new(name='TransientNodeGroup', type='ShaderNodeTree')
node_group.interface.new_socket(name='BSDF', in_out='INPUT',  socket_type='NodeSocketShader')
node_group.interface.new_socket(name='BSDF', in_out='OUTPUT', socket_type='NodeSocketShader')

node1 = node_group.nodes.new('ShaderNodeLightPath')
node1.location = (0, 0)

node2 = node_group.nodes.new('ShaderNodeMath')
node2.location = (200, 0)
node2.operation = 'POWER'

node3 = node_group.nodes.new('NodeGroupInput')
node3.location = (400, 0)

node4 = node_group.nodes.new('ShaderNodeMixShader')
node4.location = (600, 0)

node5 = node_group.nodes.new('NodeGroupOutput')
node5.location = (800, 0)

nodea = node_group.nodes.new('ShaderNodeMath')
#nodea.operation = 'COMPARE'
nodea.operation = 'LESS_THAN'

nodeb = node_group.nodes.new('ShaderNodeMath')
nodeb.operation = 'MULTIPLY'

#node_group.links.new(node1.outputs['Ray Length'], node2.inputs[1])
#node_group.links.new(node2.outputs[0], node4.inputs[0])
#node_group.links.new(node3.outputs[0], node4.inputs[2])
#node_group.links.new(node4.outputs[0], node5.inputs[0])

node_group.links.new(node1.outputs['Ray Length'], nodeb.inputs[0])
node_group.links.new(node1.outputs['Ray Depth'], nodea.inputs[1])
node_group.links.new(nodea.outputs[0], nodeb.inputs[1])
node_group.links.new(nodeb.outputs[0], node2.inputs[1])
node_group.links.new(node2.outputs[0], node4.inputs[0])
node_group.links.new(node3.outputs[0], node4.inputs[2])
node_group.links.new(node4.outputs[0], node5.inputs[0])

node2.inputs[0].default_value = 1
nodea.inputs[0].default_value = 0

In [7]:
node_group = bpy.data.node_groups.new(name='LightBounceNodeGroup', type='ShaderNodeTree')
node_group.interface.new_socket(name='BSDF', in_out='INPUT',  socket_type='NodeSocketShader')
node_group.interface.new_socket(name='BSDF', in_out='OUTPUT', socket_type='NodeSocketShader')

node1 = node_group.nodes.new('ShaderNodeLightPath')
node1.location = (0, 0)

node2 = node_group.nodes.new('ShaderNodeMath')
node2.location = (200, 0)
node2.operation = 'COMPARE'

node3 = node_group.nodes.new('NodeGroupInput')
node3.location = (400, 0)

node4 = node_group.nodes.new('ShaderNodeMixShader')
node4.location = (600, 0)

node5 = node_group.nodes.new('NodeGroupOutput')
node5.location = (800, 0)

node_group.links.new(node1.outputs['Ray Depth'], node2.inputs[1])
node_group.links.new(node2.outputs[0], node4.inputs[0])
node_group.links.new(node3.outputs[0], node4.inputs[2])
node_group.links.new(node4.outputs[0], node5.inputs[0])

node2.inputs[0].default_value = 0
node2.inputs[2].default_value = 0.1

In [8]:
# Loop through all materials
for material in bpy.data.materials:
    if material.use_nodes:
        node_tree = material.node_tree

        # Find the Material Output node
        material_output_node = None
        for node in node_tree.nodes:
            if node.bl_idname == "ShaderNodeOutputMaterial":
                material_output_node = node
                break

        if material_output_node:
            # Find the link connected to the Material Output's Surface input
            surface_link = None
            for link in node_tree.links:
                if link.to_node == material_output_node and link.to_socket.name == "Surface":
                    surface_link = link
                    break

            if surface_link:
                # Get the existing shader node connected to the Surface input
                existing_shader_node = surface_link.from_node
                existing_shader_socket = surface_link.from_socket

                # Create a Mix Shader node
                group_node = node_tree.nodes.new('ShaderNodeGroup')
                group_node.node_tree = bpy.data.node_groups['TransientNodeGroup']
                group_node.location = (material_output_node.location.x - 100, material_output_node.location.y - 200)

                # Disconnect the existing link
                node_tree.links.remove(surface_link)

                # Connect the existing shader to the first input of the Mix Shader
                node_tree.links.new(existing_shader_socket, group_node.inputs[0])

                # Connect the Mix Shader to the Material Output's Surface input
                node_tree.links.new(group_node.outputs[0], material_output_node.inputs["Surface"])

                print(f"Added Mix Shader node to material: {material.name}")
            else:
                print(f"No shader connected to Material Output in material: {material.name}")
        else:
            print(f"Material Output node not found in material: {material.name}")
    else:
        print(f"Material does not use nodes: {material.name}")

Added Mix Shader node to material: Emissive Material
Added Mix Shader node to material: Green Material
Added Mix Shader node to material: Red Material
Added Mix Shader node to material: White Material


/var/folders/fg/nxhkxc3d22s15cc0vxm03vxh0000gn/T/ipykernel_7988/3824139537.py:3: DeprecationWarning: 'Material.use_nodes' is expected to be removed in Blender 6.0
  if material.use_nodes:


In [9]:
#bpy.ops.object.light_add(type='POINT', location=cam.location)
#light = bpy.context.active_object
#light.data.energy = 1e8

light.data.use_nodes = True
nodes = light.data.node_tree.nodes
nodes.clear()

node1 = nodes.new('ShaderNodeEmission')
node1.location = (0, 0)

node2 = nodes.new('ShaderNodeGroup')
node2.location = (200, 0)
node2.node_tree = bpy.data.node_groups['TransientNodeGroup']

node3 = nodes.new('ShaderNodeGroup')
node3.location = (400, 0)
node3.node_tree = bpy.data.node_groups['LightBounceNodeGroup']

node4 = nodes.new('ShaderNodeOutputLight')
node4.location = (600, 0)

links = light.data.node_tree.links

links.new(node1.outputs[0], node2.inputs[0])
links.new(node2.outputs[0], node3.inputs[0])
links.new(node3.outputs[0], node4.inputs[0])

/var/folders/fg/nxhkxc3d22s15cc0vxm03vxh0000gn/T/ipykernel_7988/1009027171.py:5: DeprecationWarning: 'PointLight.use_nodes' is expected to be removed in Blender 6.0
  light.data.use_nodes = True


bpy.data.lights['Point'].node_tree

In [ ]:
N = 1000

bpy.context.scene.cycles.use_animated_seed = True
bpy.context.scene.frame_start = 1
bpy.context.scene.frame_end = N

path = "/tmp/output_"
path_transient = "/tmp/output_transient_"

transient = np.zeros((256,256,256,4))
indy, indx = np.meshgrid(range(256),range(256))

nodea.inputs[0].default_value = 0

for bounce in range(3):
    bpy.data.node_groups['LightBounceNodeGroup'].nodes['Math'].inputs[0].default_value = bounce+1

    bpy.context.scene.render.filepath = path
    bpy.data.node_groups['TransientNodeGroup'].nodes['Math'].inputs[0].default_value = 1
    bpy.ops.render.render(animation=True)

    bpy.context.scene.render.filepath = path_transient
    bpy.data.node_groups['TransientNodeGroup'].nodes['Math'].inputs[0].default_value = 0.9999
    bpy.ops.render.render(animation=True)

    for itr in range(N):
        render = bpy.data.images.load(path + "{:04}".format(itr+1) + ".exr")
        width, height = render.size
        pixels = np.array(render.pixels)
        pixels = pixels.reshape((height, width, 4))

        render = bpy.data.images.load(path_transient + "{:04}".format(itr+1) + ".exr")
        width, height = render.size
        pixels_transient = np.array(render.pixels)
        pixels_transient = pixels_transient.reshape((height, width, 4))

        depth = -np.log(pixels_transient / (pixels + 1e-9))
        depth = np.min(depth[:,:,0:3], axis=2)
        depth = 2.0*depth
        indz = np.clip(np.round(255*depth).astype(int),0,255)

        transient[indx[:],indy[:],indz[:],:] += 1e2 * pixels[:] / N

frame = bpy.data.images.new('src', 256, 256)
for itr in range(256):
    frame.pixels = transient[:,:,itr,:].ravel()
    frame.filepath_raw = './tmp/bounce_' + str(itr) + '.exr'
    frame.file_format = 'OPEN_EXR'
    frame.save()


00:47.594  render           | Saved: '/tmp/output_0001.exr'
00:47.603  render           | Saved: '/tmp/output_0002.exr'
00:47.611  render           | Saved: '/tmp/output_0003.exr'
00:47.620  render           | Saved: '/tmp/output_0004.exr'
00:47.629  render           | Saved: '/tmp/output_0005.exr'
00:47.638  render           | Saved: '/tmp/output_0006.exr'
00:47.646  render           | Saved: '/tmp/output_0007.exr'
00:47.655  render           | Saved: '/tmp/output_0008.exr'
00:47.664  render           | Saved: '/tmp/output_0009.exr'
00:47.673  render           | Saved: '/tmp/output_0010.exr'
00:47.682  render           | Saved: '/tmp/output_0011.exr'
00:47.691  render           | Saved: '/tmp/output_0012.exr'
00:47.699  render           | Saved: '/tmp/output_0013.exr'
00:47.708  render           | Saved: '/tmp/output_0014.exr'
00:47.717  render           | Saved: '/tmp/output_0015.exr'
00:47.725  render           | Saved: '/tmp/output_0016.exr'
00:47.734  render           | Saved: '/t

/var/folders/fg/nxhkxc3d22s15cc0vxm03vxh0000gn/T/ipykernel_7988/1708418909.py:37: RuntimeWarning: divide by zero encountered in log
  depth = -np.log(pixels_transient / (pixels + 1e-9))
/var/folders/fg/nxhkxc3d22s15cc0vxm03vxh0000gn/T/ipykernel_7988/1708418909.py:40: RuntimeWarning: invalid value encountered in cast
  indz = np.clip(np.round(255*depth).astype(int),0,255)


01:23.099  render           | Saved: '/tmp/output_0001.exr'
01:23.109  render           | Saved: '/tmp/output_0002.exr'
01:23.118  render           | Saved: '/tmp/output_0003.exr'
01:23.128  render           | Saved: '/tmp/output_0004.exr'
01:23.137  render           | Saved: '/tmp/output_0005.exr'
01:23.147  render           | Saved: '/tmp/output_0006.exr'
01:23.158  render           | Saved: '/tmp/output_0007.exr'
01:23.167  render           | Saved: '/tmp/output_0008.exr'
01:23.177  render           | Saved: '/tmp/output_0009.exr'
01:23.187  render           | Saved: '/tmp/output_0010.exr'
01:23.196  render           | Saved: '/tmp/output_0011.exr'
01:23.205  render           | Saved: '/tmp/output_0012.exr'
01:23.215  render           | Saved: '/tmp/output_0013.exr'
01:23.224  render           | Saved: '/tmp/output_0014.exr'
01:23.233  render           | Saved: '/tmp/output_0015.exr'
01:23.242  render           | Saved: '/tmp/output_0016.exr'
01:23.251  render           | Saved: '/t

: 